# Pydantic格式的使用


In [51]:
from itertools import tee
from tempfile import tempdir
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

from numpy import extract

load_dotenv(override=True)
ZHIPU_API_KEY = os.getenv("ZHIPU_API_KEY")
ZHIPU_BASE_URL = os.getenv("ZHIPU_BASE_URL")

model=init_chat_model(
    model="glm-5.2",  # 模型名称
    model_provider="openai",
    api_key=ZHIPU_API_KEY,
    base_url=ZHIPU_BASE_URL,  # ZHIPU API 的基础 URL
    extra_body={
        "thinking": {
            "type": "disabled"
        }
    }
)

In [ ]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

load_dotenv(override=True)
TONGYI_API_KEY = os.getenv("TONGYI_API_KEY")
TONGYI_BASE_URL = os.getenv("TONGYI_BASE_URL")

model=init_chat_model(
    model="deepseek-v4-pro",  # 模型名称
    model_provider="openai",
    api_key=TONGYI_API_KEY,
    base_url="https://llm-v2xyqhn297xtv6xq.cn-beijing.maas.aliyuncs.com/compatible-mode/v1",  # TONGYI API 的基础 URL

)

In [54]:

from pydantic import BaseModel,Field


class Person(BaseModel):
    """人物信息"""
    name: str = Field(description="姓名")
    age : int = Field(description="年龄")
    occupation: str = Field(description="职业")

# 创建结构化输出的大语言模型
structured_model = model.with_structured_output(Person,method="function_calling")

result = structured_model.invoke("张三是一名30岁的软件工程师")

print(result)
print(type(result))

name='张三' age=30 occupation='软件工程师'
<class '__main__.Person'>


In [23]:

from typing import TypedDict


class Person(TypedDict):
    """人物信息"""
    name: str
    age: int
    occupation: str
    
structured_model = model.with_structured_output(Person,method="json_schema")

result = structured_model.invoke("请提取人物信息，直接输出纯JSON（禁止使用```json等markdown代码块包裹）：张三是一名30岁的软件工程师")

print(result)
print(type(result))

{'name': '张三', 'age': 30, 'occupation': '软件工程师'}
<class 'dict'>


In [26]:
class MovieModel(BaseModel):
    """电影的详细信息"""
    title : str = Field(description="电影标题")
    year : int = Field(description="发行年份")
    director : str = Field(description="导演")
    rating : float = Field(description="电影评分，满分十分")


structured_model = model.with_structured_output(MovieModel)

result = structured_model.invoke("给出电影盗梦空间的信息")
print(result)

title='盗梦空间' year=2010 director='克里斯托弗·诺兰' rating=9.3


In [27]:
from pydantic import BaseModel, Field
# 定义输出结构
class SentimentAnalysis(BaseModel):
    """情感分析结果"""
    sentiment: str = Field(description="情感倾向：positive/negative/neutral")
    confidence: float = Field(description="置信度，0-1之间")
    keywords: list[str] = Field(description="关键词列表")


# ✅ v1.x：使用 with_structured_output
structured_model = model.with_structured_output(SentimentAnalysis)

# 调用
text = "这个课程内容很实用，学到了很多知识，强烈推荐！"
result = structured_model.invoke(
    f"分析以下文本的情感：\n{text}"
)

print(f"类型: {type(result)}")  # <class 'SentimentAnalysis'>
print(f"情感: {result.sentiment}")
print(f"置信度: {result.confidence}")
print(f"关键词: {result.keywords}")

类型: <class '__main__.SentimentAnalysis'>
情感: positive
置信度: 0.95
关键词: ['实用', '学到了很多', '强烈推荐', '正面']
